In [1]:
# Resolve the CDR dataset and the workspace bucket
cdr    <- trimws(tail(system("wb resource resolve --id=C2024Q3R9", intern = TRUE), 1))
bucket <- trimws(tail(system("wb resource resolve --id=rw-migration-aou-rw-6f1dba72", intern = TRUE), 1))

cdr     # expect something like "project-id.C2024Q3R9" (exactly one dot)
bucket  # expect "gs://..."
Sys.setenv(WORKSPACE_CDR = cdr, WORKSPACE_BUCKET = bucket)
cat("CDR   :", cdr, "\nbucket:", bucket, "\n")        # wb-silky-artichoke-2408.C2024Q3R9 | gs://rw-migration-aou-rw-6f1dba72


[1] "wb-silky-artichoke-2408.C2024Q3R9"

[1] "gs://rw-migration-aou-rw-6f1dba72"

CDR   : wb-silky-artichoke-2408.C2024Q3R9 
bucket: gs://rw-migration-aou-rw-6f1dba72 


In [2]:
# ==== CELL 2
library(data.table)
BASE        <- "/home/jupyter/workspace/rw-migration-aou-rw-6f1dba72/workspaces"
ROOT        <- file.path(BASE, "temporaldigitalhealthphenotyping")           # = Allofus/ of the repo
EXPORT_DATE <- "20261031"
OUT_DIR     <- file.path(BASE, "result/selection", EXPORT_DATE)
OLD_PID     <- file.path(BASE, "result/final/final_pid.txt")                 # the earlier selection (R8), for comparison
dir.create(file.path(OUT_DIR, "units"), recursive = TRUE, showWarnings = FALSE)
source(file.path(ROOT, "final_missing/R/utils.R"))                            # age_on(), log_info()
source(file.path(ROOT, "functions/cci_function.R"))                          # cci_calc_func()
source(file.path(ROOT, "selection/selection_sql.R"))
source(file.path(ROOT, "selection/selection_core.R"))
stopifnot("GOOGLE_PROJECT is empty: set it (billing project)" = nzchar(Sys.getenv("GOOGLE_PROJECT")))


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::between()     masks data.table::between()
✖ dplyr::filter()      masks stats::filter()
✖ dplyr::first()       masks data.table::first()
✖ lubridate::hour()    masks data.table::hour()
✖ lubridate::isoweek() masks data.table::isoweek()
✖ lubridate::isoyear() masks data.table::isoyear()
✖ dplyr::lag()         masks stats::lag()
✖ dplyr::last()        masks data.table::last()
✖ lubridate::mday()    masks data.table::mday()
✖ lubridate::minute()  masks data.table::minute()
✖ lubridate::month()   masks data.table::month()
✖ lubridate::quarter() masks data.table::quarter()
✖ lubridate::second()  masks data.table::second()
✖ purrr::transpose()  

In [7]:
# ==== CELL 3   (the pool of every cohort and date of birth)
persons <- rbindlist(lapply(names(SEL_COHORTS), function(co) {
  log_info("pool %s", co); x <- sel_fetch(sel_sql_person(co)); x[, cohort := co]
}))
persons[, `:=`(dob = as.Date(date_of_birth), date_of_birth = NULL)]
print(persons[, .N, by = cohort])


06:17:23 | pool Diabetes

06:17:28 | pool Insomnia

06:17:31 | pool Prostate

06:17:34 | pool Healthy



     cohort     N
     <char> <int>
1: Diabetes  4783
2: Insomnia  5084
3: Prostate   627
4:  Healthy 22970


In [12]:
# ==== CELL 4   (wear metrics, SQL; one file per cohort x stream)
wear <- rbindlist(lapply(names(SEL_COHORTS), function(co) rbindlist(lapply(names(SEL_STREAMS), function(s) {
  f <- file.path(OUT_DIR, "units", sprintf("wear_%s_%s.tsv", co, s))
  if (!file.exists(f)) { log_info("wear %s / %s", co, s); fwrite(sel_fetch(sel_sql_wear(co, s)), f, sep = "\t") }
  x <- fread(f); x[, `:=`(cohort = co, stream = s)]
}))))
wear <- sel_prepare_wear(wear)
fwrite(wear, file.path(OUT_DIR, "wear_metrics.tsv"), sep = "\t")
print(wear[, .(n_people = .N, median_days = as.numeric(median(n_days)), median_pct_wear = round(median(pct_wear), 1)),
           by = .(cohort, stream)])


      cohort   stream n_people median_days median_pct_wear
      <char>   <char>    <int>       <num>           <num>
 1: Diabetes activity     4783       247.0            95.0
 2: Diabetes    sleep     4783       102.0            66.9
 3: Diabetes       hr     4783       145.0            86.6
 4: Insomnia activity     5084       348.0            93.9
 5: Insomnia    sleep     5084       124.0            68.6
 6: Insomnia       hr     5084       201.0            86.3
 7: Prostate activity      627       149.0            98.0
 8: Prostate    sleep      627       100.0            81.6
 9: Prostate       hr      627       115.0            92.6
10:  Healthy activity    22970       534.5            96.2
11:  Healthy    sleep    22970       189.0            69.1
12:  Healthy       hr    22970       331.0            90.0


In [13]:
# ==== CELL 5   (check: the SQL result equals the metrics recomputed from the raw dates of a random sample)
sel_verify(wear, function(s, ids) sel_fetch(sel_sql_dates(s, ids)), n = 30L)


     stream n_checked n_mismatch
     <char>     <int>      <int>
1: activity        30          0
2:    sleep        30          0
3:       hr        30          0


In [14]:
# ==== CELL 6   (age at the first record over the three streams, conditions, CCI)
persons <- sel_persons(persons, wear)
print(persons[, .(n = .N, no_record = sum(is.na(first_date)), median_age = as.numeric(median(age_first, na.rm = TRUE))), by = cohort])
codes <- sel_cci_codes(); cat(length(codes), "SNOMED codes in the CCI lookup\n")
conds <- rbindlist(lapply(names(SEL_COHORTS), function(co) {
  log_info("conditions %s", co); x <- sel_fetch(sel_sql_conditions(co, codes)); x[, cohort := co]
}))
cci <- sel_cci(persons, conds)
fwrite(persons, file.path(OUT_DIR, "persons.tsv"), sep = "\t")
fwrite(cci, file.path(OUT_DIR, "cci.txt"), sep = "\t", quote = FALSE)       # same columns as pre_final_cci.txt
print(cci[, .(n = .N, median_cci = median(cci_score), n_above_10 = sum(cci_score > 10)), by = cohort])


Key: <cohort>
     cohort     n no_record median_age
     <char> <int>     <int>      <num>
1: Diabetes  4783         0         58
2:  Healthy 22970         0         48
3: Insomnia  5084         0         56
4: Prostate   627         0         71
785 SNOMED codes in the CCI lookup


06:30:48 | conditions Diabetes

06:30:52 | conditions Insomnia

06:30:55 | conditions Prostate

06:30:58 | conditions Healthy



     cohort     n median_cci n_above_10
     <char> <int>      <num>      <int>
1: Diabetes  4783          5        369
2: Insomnia  5084          3        238
3: Prostate   627          4         59
4:  Healthy 22970          0          0


In [15]:
# ==== CELL 7   (the selection with the default rules, and its comparison with the earlier final_pid.txt)
pool <- persons[, .(cohort, person_id)]
sel  <- sel_write(sel_select(wear, cci, pool), OUT_DIR, "base")
print(sel$attrition)
old <- fread(OLD_PID)
print(sel_compare(sel$pids, old))
# Do the earlier people pass the wear rules (CCI left out)? only_old = people the wear rules do not reproduce.
print(sel_compare(sel_select(wear, cci, pool, cci_max = Inf)$pids, old)[, .(cohort, n_old, only_old)])


      cohort                      step     n
      <char>                    <fctr> <int>
 1: Diabetes                      pool  4783
 2: Diabetes data in activity+sleep+hr  4783
 3: Diabetes                days >= 21  4140
 4: Diabetes         longest gap <= 14  1644
 5: Diabetes   compliance in [70, Inf)  1425
 6: Diabetes                 CCI <= 10  1306
 7:  Healthy                      pool 22970
 8:  Healthy data in activity+sleep+hr 22970
 9:  Healthy                days >= 21 20362
10:  Healthy         longest gap <= 14  7167
11:  Healthy   compliance in [70, Inf)  6381
12:  Healthy                 CCI <= 10  6381
13: Insomnia                      pool  5084
14: Insomnia data in activity+sleep+hr  5084
15: Insomnia                days >= 21  4469
16: Insomnia         longest gap <= 14  1668
17: Insomnia   compliance in [70, Inf)  1457
18: Insomnia                 CCI <= 10  1388
19: Prostate                      pool   627
20: Prostate data in activity+sleep+hr   627
21: Prosta

In [16]:
# ==== CELL 8   (another rule set: only the argument you name changes; example = compliance 50-60%, i.e. 50 <= % < 60)
sel_write(sel_select(wear, cci, pool, comp_min = 50, comp_max = 60), OUT_DIR, "comp50-60")
#other arguments: min_days = 21, gap_max = 14, comp_min = 70, comp_max = Inf, cci_max = 10


In [20]:
# ==== CELL 9   (export the selected people; the missingness pipeline is only read)
source(file.path(ROOT, "final_missing/load_missingness.R")); load_missingness(file.path(ROOT, "final_missing"))
cfg <- make_config(list(export = list(date = EXPORT_DATE)))
final_pid <- read_final_pid(file.path(OUT_DIR, "final_pid_base.txt"))
print(final_pid[, .N, by = cohort])
exports <- run_bq_exports(cfg, final_pid)       # to <bucket>/bq_exports/<owner>/20261031/
# The ETL then runs with EXPORT_DATE "20261031", FINAL_PID = final_pid_base.txt and CCI_FILE = cci.txt of OUT_DIR.

     cohort     N
     <char> <int>
1: Diabetes  1306
2:  Healthy  6381
3: Insomnia  1388
4: Prostate   253


07:02:12 | export Insomnia/activity: running query for 1388 people

Auto-refreshing stale OAuth token.

07:02:24 | export Insomnia/sleep: running query for 1388 people

07:02:34 | export Insomnia/hr: running query for 1388 people

07:02:46 | export Insomnia/meta: running query for 1388 people

07:02:53 | export Diabetes/activity: running query for 1306 people

07:03:04 | export Diabetes/sleep: running query for 1306 people

07:03:13 | export Diabetes/hr: running query for 1306 people

07:03:25 | export Diabetes/meta: running query for 1306 people

07:03:30 | export Healthy/activity: running query for 6381 people

07:03:42 | export Healthy/sleep: running query for 6381 people

07:03:58 | export Healthy/hr: running query for 6381 people

07:04:35 | export Healthy/meta: running query for 6381 people

07:04:43 | export Prostate/activity: running query for 253 people

07:04:50 | export Prostate/sleep: running query for 253 people

07:04:56 | export Prostate/hr: running query for 253 people


      cohort   stream n_people   status      note
      <char>   <char>    <int>   <char>    <char>
 1: Insomnia activity     1388 exported 1 file(s)
 2: Insomnia    sleep     1388 exported 1 file(s)
 3: Insomnia       hr     1388 exported 1 file(s)
 4: Insomnia     meta     1388 exported 1 file(s)
 5: Diabetes activity     1306 exported 1 file(s)
 6: Diabetes    sleep     1306 exported 1 file(s)
 7: Diabetes       hr     1306 exported 1 file(s)
 8: Diabetes     meta     1306 exported 1 file(s)
 9:  Healthy activity     6381 exported 6 file(s)
10:  Healthy    sleep     6381 exported 1 file(s)
11:  Healthy       hr     6381 exported 1 file(s)
12:  Healthy     meta     6381 exported 1 file(s)
13: Prostate activity      253 exported 1 file(s)
14: Prostate    sleep      253 exported 1 file(s)
15: Prostate       hr      253 exported 1 file(s)
16: Prostate     meta      253 exported 1 file(s)


In [3]:
# ==== CELL 10   (summary tables and Venn diagram of the selection in OUT_DIR; needs Cells 1-2 and the files of Cells 4-7)
# tables/table1_demographics_<tag>_{all,excl_overlap}.tsv, tables/table_fitbit_<tag>_{all,excl_overlap}.tsv, tables/venn_counts_<tag>.tsv,
# figures/venn_<tag>.png / .pdf.   all = every cohort membership (N = rows of final_pid); excl_overlap = people in several cohorts removed.
source(file.path(ROOT, "final_missing/R/persons.R"))                        # race_group_from_name()
source(file.path(ROOT, "selection/summary_tables.R"))
TAG  <- "base"                                                              # or "comp50-60" etc.
meta <- sel_fetch(st_sql_meta(unique(fread(file.path(OUT_DIR, sprintf("final_pid_%s.txt", TAG)))$person_id)))
res  <- st_run(OUT_DIR, TAG, meta)
print(res$all$table1); print(res$all$fitbit); print(res$venn)


                                    Demographic    Insomnia    Diabetes
                                         <char>      <char>      <char>
 1:                                           N       1,388       1,306
 2:                                         Age                        
 3:                                       18-44   323 (23%)   205 (16%)
 4:                                       45-64   495 (36%)   517 (40%)
 5:                                        >=65   570 (41%)   583 (45%)
 6:                                   Ethnicity                        
 7:                          Hispanic or Latino     97 (7%)    107 (8%)
 8:                      Not Hispanic or Latino 1,267 (91%) 1,163 (89%)
 9:                                      Others     24 (2%)     36 (3%)
10:                                        Race                        
11:            American Indian or Alaska Native     10 (1%)      8 (1%)
12:                                       Asian     25 (2%)     